In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re

## Load Cached Data from Local Storage

In [71]:
# META DATA CSV 

# check needed columns
metadata_cols = pd.read_csv('data/git_data/metadata.csv', nrows=0).columns
print(metadata_cols)

# load metadata as df
meta_df = pd.read_csv('data/git_data/metadata.csv',
                      usecols=['building_id', 'site_id', 'primaryspaceusage', 'electricity', 'sqm'],
                      index_col='building_id'
                     )

Index(['building_id', 'site_id', 'building_id_kaggle', 'site_id_kaggle',
       'primaryspaceusage', 'sub_primaryspaceusage', 'sqm', 'sqft', 'lat',
       'lng', 'timezone', 'electricity', 'hotwater', 'chilledwater', 'steam',
       'water', 'irrigation', 'solar', 'gas', 'industry', 'subindustry',
       'heatingtype', 'yearbuilt', 'date_opened', 'numberoffloors',
       'occupants', 'energystarscore', 'eui', 'site_eui', 'source_eui',
       'leed_level', 'rating'],
      dtype='str')


In [74]:
# filter building to include only Office or Education buildings

meta_df = meta_df[(meta_df['site_id'] == 'Panther') & 
    (meta_df['primaryspaceusage'].isin(['Education', 'Office'])) &
    (meta_df['electricity'] == 'Yes')]

print(meta_df.shape)
meta_df.head()

(54, 4)


,site_id,primaryspaceusage,sqm,electricity
building_id,,,,
Panther_education_Rosalie,Panther,Education,690.5,Yes
Panther_education_Misty,Panther,Education,252.7,Yes
Panther_education_Mattie,Panther,Education,499.4,Yes
Panther_education_Diann,Panther,Education,2200.4,Yes
Panther_education_Gina,Panther,Education,10833.1,Yes


In [30]:
# ELECTRICITY CSV

# check the electrical buildings csv for specific columns
elec_cols = pd.read_csv('data/git_data/electricity.csv', nrows=0).columns
print(len(elec_cols))

# search for number of columns with Panther and (Education or Office)
# include the timestamp column for analysis
search_pattern = r'^timestamp$|Panther.*(?:education|office)'
matched_cols = elec_cols[elec_cols.str.contains(search_pattern, regex=True)]

print(len(matched_cols))

1579
55


In [67]:
# load electric consumption profiles of education and office buildings
elec_df = pd.read_csv('data/git_data/electricity.csv',
                      index_col=0,
                      usecols=matched_cols,
                      parse_dates=True,
                     )

print(elec_df.shape)
elec_df.sample(10)

(17544, 54)


,Panther_office_Hannah,Panther_education_Teofila,Panther_education_Jerome,Panther_education_Misty,Panther_office_Catherine,Panther_education_Tina,Panther_education_Janis,Panther_office_Patti,Panther_office_Lauretta,Panther_office_Valarie,...,Panther_education_Diann,Panther_education_Emily,Panther_education_Scarlett,Panther_education_Zelda,Panther_office_Jeane,Panther_office_Lavinia,Panther_office_Lois,Panther_education_Gina,Panther_education_Karri,Panther_education_Cleopatra
timestamp,,,,,,,,,,,,,,,,,,,,,
2016-08-20 00:00:00,8.5826,95.0133,458.1884,41.2880,19.4037,12.3024,21.7362,413.6798,188.1963,20.1639,...,136.9264,74.0143,17.1633,49.6096,20.7640,22.4043,142.7876,460.8889,555.9473,6.8413
2016-04-10 05:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2016-02-13 13:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2016-04-30 14:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2016-01-07 05:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2017-02-09 13:00:00,12.6885,210.9527,546.8055,36.0870,43.5684,8.3016,41.4430,485.6937,180.5148,38.8075,...,119.0230,144.4279,32.8463,105.6204,26.1651,33.5265,176.9942,485.4937,374.9524,42.2481
2017-07-14 03:00:00,6.8913,102.9459,469.4906,40.3278,17.0833,9.2018,23.5605,372.0718,176.1940,28.5655,...,24.5047,79.2153,15.3030,41.6080,13.4426,17.0433,164.7318,407.8787,636.1227,11.7623
2016-01-14 18:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2016-06-16 18:00:00,7.0004,145.8781,505.3975,37.9273,47.8492,12.4024,29.4297,583.3126,192.2771,15.3230,...,145.2280,139.6269,25.4849,84.8164,30.2458,21.4442,117.2226,507.8980,371.5917,15.6030


In [38]:
# WEATHER CSV

# filter the columns of the weather csv

w_cols = pd.read_csv('data/git_data/weather.csv', nrows=0).columns
print(w_cols)

# site_id must correspond to Panther site
df_weather = pd.read_csv('data/git_data/weather.csv', 
                         index_col=0,
                         usecols=['timestamp', 'site_id', 'airTemperature', 'dewTemperature',],
                         parse_dates=True
                        )
df_weather = df_weather[df_weather['site_id']=='Panther']
print(df_weather.shape)
df_weather.head()

Index(['timestamp', 'site_id', 'airTemperature', 'cloudCoverage',
       'dewTemperature', 'precipDepth1HR', 'precipDepth6HR', 'seaLvlPressure',
       'windDirection', 'windSpeed'],
      dtype='str')
(17544, 3)


,site_id,airTemperature,dewTemperature
timestamp,,,
2016-01-01 00:00:00,Panther,19.4,19.4
2016-01-01 01:00:00,Panther,21.1,21.1
2016-01-01 02:00:00,Panther,21.1,21.1
2016-01-01 03:00:00,Panther,20.6,20.0
2016-01-01 04:00:00,Panther,21.1,20.6


## Normalize the Building Electrical Profiles

To enable a fair comparison across buildings of different sizes, divide the hourly electricity consumption ($\text{kWh}$) by the gross floor area ($\text{m}^2$) to calculate the hourly specific electrical intensity ($\text{kWh/m}^2$):

$$e_t = \frac{E_t}{\text{Gross Floor Area}} \quad \left[\frac{\text{kWh}}{\text{m}^2}\right]$$

Where:
- $E_t$ is the electricity consumed at hour $t$ ($\text{kWh}$)
- $e_t$ is the area-normalized electricity intensity at hour $t$ ($\text{kWh/m}^2$)

In [80]:
# apply meta_df sqm across elec_df to normalize the buildings
meta_sqm = meta_df['sqm']
elec_normalized = elec_df.div(meta_sqm, axis=1)
elec_normalized.sample(5)

,Panther_education_Alecia,Panther_education_Annetta,Panther_education_Aurora,Panther_education_Cleopatra,Panther_education_Diann,Panther_education_Edna,Panther_education_Emily,Panther_education_Enriqueta,Panther_education_Genevieve,Panther_education_Gina,...,Panther_office_Larry,Panther_office_Lauretta,Panther_office_Lavinia,Panther_office_Lois,Panther_office_Otto,Panther_office_Patti,Panther_office_Ruthie,Panther_office_Shauna,Panther_office_Taryn,Panther_office_Valarie
timestamp,,,,,,,,,,,,,,,,,,,,,
2017-03-17 13:00:00,0.028684,0.013021,0.017750,0.022728,0.050091,0.016979,0.027510,0.021324,0.020720,0.043135,...,0.046659,0.042779,0.038513,0.147748,0.039878,0.032827,0.030685,0.029633,0.016411,0.033406
2016-08-03 11:00:00,0.032764,0.014538,0.019472,0.020123,0.073410,0.016454,0.022135,0.016625,0.022357,0.049451,...,0.045095,0.063883,0.028446,0.104631,NaN,0.038509,0.028667,0.028299,0.020159,0.025330
2017-03-01 02:00:00,0.015464,0.008504,0.006690,0.010857,0.045682,0.010551,0.011067,0.007635,0.019067,0.039461,...,0.024112,0.047556,0.020935,0.140160,0.019681,0.021958,0.017884,0.011836,0.012462,0.023296
2016-07-20 01:00:00,0.018034,0.010237,0.010510,0.003977,0.057955,0.008651,0.011890,0.008990,0.018039,0.044650,...,0.028989,0.051834,0.018777,0.101056,NaN,0.028965,0.024938,0.011613,0.018677,0.017225
2017-01-08 13:00:00,0.028928,0.014863,0.018024,0.025432,0.029682,0.012694,0.035795,0.024983,0.021247,0.037226,...,0.038929,0.034936,0.038114,0.143668,0.034528,0.030565,0.033037,0.012636,0.012995,0.039562
